Same model, but fine tuning on this <prompt> english <ans> genz <eot>. 
GenZ version of English

In [1]:
import tiktoken
from datasets import load_dataset
import numpy as np

In [2]:
base = tiktoken.get_encoding("gpt2")
enc = tiktoken.Encoding(
    name="gpt2_answer",
    pat_str=base._pat_str,
    mergeable_ranks=base._mergeable_ranks,
    special_tokens={**base._special_tokens, "<answer>": base.n_vocab},
)
ANS, EOT = enc.encode_single_token("<answer>"), enc.eot_token

In [3]:
ds = load_dataset("Programmer-RD-AI/genz-slang-pairs-1k",  num_proc=8);

ids = []
for row in ds["train"]:
	eng = row["normal"]
	slang = row["gen_z"]
	curr_ids = enc.encode_ordinary(eng) + [ANS] + enc.encode_ordinary(slang) + [EOT]
	ids.append(curr_ids)

In [4]:
ds = load_dataset("GCruz19/GenZ_data",  num_proc=8);

for row in ds["train"]:
	try:
		entry = json.loads(row['translation'])
		eng = entry['normal']
		slang = entry['genz']
		curr_ids = eng + [ANS] + enc.encode_ordinary(slang) + [EOT]
		ids.append(curr_ids)
	except Exception as e:
		print(f"Error processing row: {row}. Error: {e}")

Error processing row: {'id': '0', 'translation': '{ "normal": "Hey, did you see Sarah\'s new outfit? It looks really great!", "genz": "Hey, did you see Sarah\'s new fit? It\'s totally fire!" }'}. Error: name 'json' is not defined
Error processing row: {'id': '1', 'translation': '{ "normal": "Oh my gosh, that concert was excellent! The atmosphere was fantastic!", "genz": "OMG, that concert was lit AF! The vibes were so on point!" }'}. Error: name 'json' is not defined
Error processing row: {'id': '2', 'translation': '{ "normal": "I can\'t believe all the drama at school today. It\'s so excessive and unnecessary!", "genz": "I can\'t even with all the drama at school today. It\'s so extra!" }'}. Error: name 'json' is not defined
Error processing row: {'id': '3', 'translation': '{ "normal": "I just found out I got the job! Let\'s celebrate together with our friends!", "genz": "Just found out I got the job! Time to celebrate with my squad!" }'}. Error: name 'json' is not defined
Error proce

In [5]:
arr = np.array(ids, dtype=object)
ixs = np.random.permutation(len(arr))
flat = np.concatenate(arr[ixs]).astype(np.uint16)
print(len(flat))
flat.tofile("genz_trans_finetune.bin")

28647


In [6]:
import os
file = f"genz_trans_finetune.bin"
n = os.path.getsize(file) // 2
print(f"{file}: {n:,} tokens, {n/1e6:.2f}M tokens")

genz_trans_finetune.bin: 28,647 tokens, 0.03M tokens
